<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Random_Forest_Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import(classification_report, confusion_matrix, accuracy_score, precision_score,
                            recall_score, f1_score, roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


In [3]:
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [4]:
def generate_spam_dataset(n_samples=2000, random_state=42):
  np.random.seed(random_state)

  #Initialize lists to store features and labels
  features=[]
  labels=[]

  for i in range(n_samples):
    #Determine if email is spam (40% spam rate)
    is_spam=np.random.random()<0.4

    if is_spam:
      #Spam characteristics
      word_count=np.random.normal(150, 50) #longer emails
      capital_ratio=np.random.beta(3, 2)*50 #More capitals
      exclamation_count=np.random.poisson(4)+1 #more exclamations
      url_count=np.random.poisson(2)+1 #more URLs
      special_char_ratio=np.random.beta(2, 3)* 30 #More special chars
      digit_ratio=np.random.beta(2, 3)* 25 #more digits
      avg_word_length=np.random.normal(4.5, 1) #Shorter words
      sentence_count=np.random.poisson(8)+3 #More sentences
    else:
      #Ham (legitimate email) characteristics
      word_count=np.random.normal(100, 40) #Shorter emails
      capital_ratio=np.random.beta(1, 4)*15 #Fewer capitals
      exclamation_count=np.random.poisson(0.5) #Fewer exclamations
      url_count=np.random.poisson(0.3) #Fewer URLs
      special_char_ratio=np.random.beta(1, 4)*10 #Fewer special chars
      digit_ratio=np.random.beta(1, 3)*15 #Fewer digits
      avg_word_length=np.random.normal(5.2, 0.8) #Longer words
      sentence_count=np.random.poisson(5)+2 #Fewer sentences

    #Ensure realistic bounds
    word_count=max(10, word_count)
    capital_ratio=np.clip(capital_ratio, 0, 100)
    exclamation_count=max(0, exclamation_count)
    url_count=max(0, url_count)
    special_char_ratio=np.clip(special_char_ratio, 0, 100)
    digit_ratio=np.clip(digit_ratio, 0, 100)
    avg_word_length=np.clip(avg_word_length, 2, 15)
    sentence_count=max(1, sentence_count)

    features.append([
        word_count, capital_ratio, exclamation_count, url_count,
        special_char_ratio, digit_ratio, avg_word_length, sentence_count
    ])
    labels.append(1 if is_spam else 0)

  #Create DataFrame
  feature_names=[
      'word_count', 'capital_ratio', 'exclamation_count', 'url_count',
      'special_char_ratio', 'digit_ratio', 'avg_word_length', 'sentence_count'
  ]

  df=pd.DataFrame(features, columns=feature_names)
  df['is_spam']=labels

  return df


In [5]:
print("Generating spam dataset...")
df=generate_spam_dataset(n_samples=2000)
print("Dataset created with {len(df)} samples")
print(f"Spam rate: {df['is_spam'].mean():.1%}")

print("\n First 5 rows of the dataset:")
df.head()

Generating spam dataset...
Dataset created with {len(df)} samples
Spam rate: 40.1%

 First 5 rows of the dataset:


,word_count,capital_ratio,exclamation_count,url_count,special_char_ratio,digit_ratio,avg_word_length,sentence_count,is_spam
0,94.405994,20.465932,5,3,20.487184,6.620101,4.274224,8,1
1,75.974452,4.031774,0,0,0.106701,1.503050,4.137451,9,0
2,159.843062,39.746444,4,2,10.979089,11.090095,2.991847,10,1
3,143.985879,2.446308,0,0,0.058800,0.484837,6.033993,7,0
4,175.072417,41.230643,5,2,27.128912,9.562667,4.333939,12,1


In [7]:
print("\n"+"="*50)
print("EDA")
print("="*50)

print("\nDataset Summary:")
print(df.describe())

print(f"\nClass Distribution:")
print(f"Ham (legitimate): {(df['is_spam']==0).sum()} ({(df['is_spam']==0).mean():.1%})")
print(f"Spam: {(df['is_spam']==1).sum()} ({(df['is_spam']==1).mean():.1%})")


EDA

Dataset Summary:
        word_count  capital_ratio  exclamation_count    url_count  \
count  2000.000000    2000.000000        2000.000000  2000.000000   
mean    118.095351      13.795597           2.314000     1.378500   
std      49.298468      14.955045           2.644598     1.683049   
min      10.000000       0.001451           0.000000     0.000000   
25%      83.892565       1.747425           0.000000     0.000000   
50%     115.569356       5.265835           1.000000     1.000000   
75%     150.926069      26.821932           4.000000     2.000000   
max     296.483884      49.176431          13.000000    11.000000   

       special_char_ratio  digit_ratio  avg_word_length  sentence_count  \
count         2000.000000  2000.000000      2000.000000     2000.000000   
mean             5.914114     6.193732         4.947229        8.600500   
std              6.223358     4.804760         0.931158        3.202972   
min              0.000518     0.000257         2.000000